# Predição dos modelos MSOV1, MSOV2 e MSOV3 nas Imagens de Mergulho

Autor:  Viviane da Rosa Sommer

Atualizado: 30/01/2025

## Objetivo:
Este notebook tem como objetivo mostrar as imagens do dataset Legado PUC, juntamente com sua predições dos modelos MSOV1,MSOV2 e MSOV3.

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch
!pip install tqdm
!pip install pandas

import glob
import cv2
import numpy as np
import os
import json
from ultralytics import YOLO
import matplotlib.pyplot as plt
import traceback 
import torchvision
import torch
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.patches as patches 
from PIL import Image 
from matplotlib.path import Path 
from sklearn.metrics import auc 
from typing import List, Dict, Any, Tuple, Optional

## Declaração de Constantes e Modelos

In [ ]:
CORAL_CLASS_ID = 0
mso_v1 = YOLO('../Coral_Note/runs/segment/Lote123-SOverlay/train-800-lote123/weights/best.pt')
mso_v2 = YOLO('../Testes-Segmentacao-PreProcessamento/runs/segment/train3/weights/best.pt')
mso_v3 = YOLO('runs/segment/train/weights/best.pt')

INPUT_DIRECTORY_POSITIVE_IMAGES = "../Coral_Note/puc_dataset/PUC - Coral - Positivo" 
INPUT_DIRECTORY_NEGATIVE_IMAGES = "../Coral_Note/puc_dataset/PUC - Coral - Negativo" 

positive_images = glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/*.[pPjJ][nNpP][gGgG]") 
negative_images = glob.glob(f"{INPUT_DIRECTORY_NEGATIVE_IMAGES}/*.[pPjJ][nNpP][gGgG]") 
 
all_images = positive_images + negative_images 
all_labels = [1] * len(positive_images) + [0] * len(negative_images) 

## Funções Necessárias

In [ ]:
def read_image_with_annotation(filename: str) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """
    Carrega uma imagem e sua máscara de anotação correspondente (se disponível) em formato JSON.

    Args:
        filename (str): Caminho para o arquivo de imagem.

    Returns:
        Tuple[np.ndarray, Optional[np.ndarray]]: Uma tupla contendo:
            - A imagem recortada como um array numpy.
            - A máscara de anotação recortada como um array numpy, ou None se não houver anotação.

    Raises:
        ValueError: Se a imagem não puder ser carregada.
    """
    # Carregar a imagem
    image = cv2.imread(filename)
    if image is None:
        raise ValueError(f"Unable to load image: {filename}")

    # Recortar a imagem
    height, width, _ = image.shape
    mid = height // 2
    top = max(0, mid - int(0.28 * height))
    bottom = min(height, mid + int(0.34 * height))
    cropped_image = image[top:bottom, :]

    # Caminho para o arquivo de anotação em JSON
    json_file = filename.replace('.jpg', '.json').replace('.png', '.json').replace('.JPG', '.json')

    # Carregar a anotação (se disponível)
    annotation_mask = None
    if os.path.exists(json_file):
        annotation_mask = load_annotations(json_file, image.shape[:2], top, bottom)

    return cropped_image, annotation_mask


def process_results(results: List[Any]) -> Optional[Any]: 
    """ 
    Extracts the first valid result containing masks from the model's output. 
 
    Args: 
        results (List[Any]): List of model detection results. 
 
    Returns: 
        Optional[Any]: The first valid result with masks, or None if no valid result exists. 
    """ 
    for result in results:
        if result.masks is not None:
            return result
    return None


def prediction_coral(img: np.ndarray, model: Any) -> np.ndarray: 
    """ 
    Predicts coral regions in an image using a given model. 
 
    Args: 
        img (np.ndarray): Input image. 
        model (Any): Model to use for prediction. 
 
    Returns: 
        np.ndarray: Binary mask for coral regions. 
    """ 
    original_height, original_width = img.shape[:2]
    coral_results = model(img, verbose=False)
    coral_best_result = process_results(coral_results)

    if coral_best_result and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_masks) > 0:
            nms_indices = torchvision.ops.nms(boxes[coral_indices, :4], coral_scores, iou_threshold=0.2)
            final_coral_mask = torch.any(coral_masks[nms_indices], dim=0).int().cpu().numpy() * 255
        else:
            final_coral_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    else:
        final_coral_mask = np.zeros((original_height, original_width), dtype=np.uint8)

    return cv2.resize(final_coral_mask, (original_width, original_height), interpolation=cv2.INTER_NEAREST)


def generate_predictions(files: List[str], model_v1: Any, model_v2: Any, model_v3: Any) -> List[Dict[str, Any]]:
    """
    Gera predições para uma lista de arquivos de imagem usando três modelos.

    Args:
        files (List[str]): Lista de caminhos para os arquivos de imagem.
        model_v1 (Any): Primeiro modelo para predição.
        model_v2 (Any): Segundo modelo para predição.
        model_v3 (Any): Terceiro modelo para predição.

    Returns:
        List[Dict[str, Any]]: Lista de dicionários contendo resultados para cada arquivo.
    """
    results = []
    for filename in files:
        try:
            cropped_image, annotation_mask = read_image_with_annotation(filename)

            mask_v1_result = prediction_coral(cropped_image, model_v1)
            mask_v2_result = prediction_coral(cropped_image, model_v2)
            mask_v3_result = prediction_coral(cropped_image, model_v3)

            mask_v1 = mask_v1_result[0] if isinstance(mask_v1_result, tuple) else mask_v1_result
            mask_v2 = mask_v2_result[0] if isinstance(mask_v2_result, tuple) else mask_v2_result
            mask_v3 = mask_v3_result[0] if isinstance(mask_v3_result, tuple) else mask_v3_result

            iou_v1 = calculate_iou(mask_v1, annotation_mask) if annotation_mask is not None else float('nan')
            iou_v2 = calculate_iou(mask_v2, annotation_mask) if annotation_mask is not None else float('nan')
            iou_v3 = calculate_iou(mask_v3, annotation_mask) if annotation_mask is not None else float('nan')

            dice_v1 = calculate_dice(mask_v1, annotation_mask) if annotation_mask is not None else float('nan')
            dice_v2 = calculate_dice(mask_v2, annotation_mask) if annotation_mask is not None else float('nan')
            dice_v3 = calculate_dice(mask_v3, annotation_mask) if annotation_mask is not None else float('nan')

            results.append({
                "file": filename,
                "cropped_image": cropped_image,
                "annotation_mask": annotation_mask,
                "mask_v1": mask_v1,
                "mask_v2": mask_v2,
                "mask_v3": mask_v3,
                "iou_v1": iou_v1,
                "iou_v2": iou_v2,
                "iou_v3": iou_v3,
                "dice_v1": dice_v1,
                "dice_v2": dice_v2,
                "dice_v3": dice_v3
            })
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            traceback.print_exc()
    return results


def load_annotations(json_path: str, image_size: Tuple[int, int], top: int, bottom: int) -> Optional[np.ndarray]:
    """
    Carrega as anotações em formato JSON e cria uma máscara binária alinhada com a imagem recortada.

    Args:
        json_path (str): Caminho para o arquivo JSON de anotação.
        image_size (Tuple[int, int]): Tamanho original da imagem (altura, largura).
        top (int): Linha superior da imagem recortada.
        bottom (int): Linha inferior da imagem recortada.

    Returns:
        Optional[np.ndarray]: Máscara binária onde as regiões anotadas são marcadas com 255,
                              ou None se não houver anotação ou se o arquivo JSON não existir.
    """
    try:
        height, width = image_size
        full_mask = np.zeros((height, width), dtype=np.uint8)

        # Verificar se o JSON existe
        if not os.path.exists(json_path):
            print(f"No annotation file found for {json_path}. Assuming negative case.")
            return None

        # Carregar o arquivo JSON
        with open(json_path, 'r') as file:
            annotation_data = json.load(file)

        # Processar os polígonos de anotação
        for shape in annotation_data['shapes']:
            if shape['shape_type'] == 'polygon':
                points = np.array(shape['points'])  # Converter os pontos para um array NumPy

                # Escalar os pontos para o tamanho original da imagem
                points[:, 0] = np.clip(points[:, 0], 0, width - 1)
                points[:, 1] = np.clip(points[:, 1], 0, height - 1)

                # Preencher a máscara completa com o polígono
                cv2.fillPoly(full_mask, [points.astype(np.int32)], 255)

        # Recortar a máscara para alinhar com a imagem recortada
        cropped_mask = full_mask[top:bottom, :]

        return cropped_mask
    except Exception as e:
        print(f"Error reading annotation file {json_path}: {e}")
        traceback.print_exc()
        return None

def plot_image_comparisons(results: List[Dict[str, Any]]) -> None:
    """
    Plota comparações das imagens recortadas com contornos das anotações e predições dos três modelos.

    Args:
        results (List[Dict[str, Any]]): Lista de resultados contendo as máscaras de predição e as anotações.

    Returns:
        None
    """
    for result in results:
        try:
            cropped_image = result["cropped_image"]
            annotation_mask = result["annotation_mask"]
            mask_v1 = result["mask_v1"]
            mask_v2 = result["mask_v2"]
            mask_v3 = result["mask_v3"]
            image_name = result["file"].split('/')[-1]  # Pegar apenas o nome do arquivo

            # Configurar o layout da figura
            fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(40, 10))

            # Função auxiliar para desenhar contornos
            def draw_contours(ax, image, mask, title, color):
                ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
                ax.axis('off')
                ax.set_title(title, fontsize=14)

                if mask is not None:
                    # Garantir que a máscara esteja binária e no formato correto
                    if mask.dtype != np.uint8:
                        mask = (mask > 0).astype(np.uint8) * 255

                    # Encontrar e desenhar os contornos
                    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    for contour in contours:
                        if len(contour) >= 3:  # Verificar se o contorno tem pelo menos 3 pontos para formar um polígono
                            path = Path(contour[:, 0, :], closed=True)
                            patch = patches.PathPatch(path, edgecolor=color, facecolor='none', lw=2)
                            ax.add_patch(patch)

            # Plotar a imagem com os contornos das anotações do especialista
            draw_contours(ax1, cropped_image, annotation_mask, 'Expert Annotation', 'red')

            # Plotar as máscaras de predição dos modelos
            draw_contours(ax2, cropped_image, mask_v1, 'Model mso_v1 Prediction', 'blue')
            draw_contours(ax3, cropped_image, mask_v2, 'Model mso_v2 Prediction', 'purple')
            draw_contours(ax4, cropped_image, mask_v3, 'Model mso_v3 Prediction', 'green')

            # Adicionar o título principal com o nome da imagem
            fig.suptitle(f"Image: {image_name}", fontsize=16)

            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"Error plotting for {result['file']}: {e}")
            traceback.print_exc()
            continue

def evaluate_predictions(results: List[Dict[str, Any]], labels: List[int]) -> None:
    """
    Avalia as predições dos três modelos usando matrizes de confusão e relatórios de classificação.

    Args:
        results (List[Dict[str, Any]]): Lista de resultados contendo predições e anotações.
        labels (List[int]): Rótulos verdadeiros para cada imagem.

    Returns:
        None
    """
    predictions_v1 = [1 if result["mask_v1"].any() else 0 for result in results]
    predictions_v2 = [1 if result["mask_v2"].any() else 0 for result in results]
    predictions_v3 = [1 if result["mask_v3"].any() else 0 for result in results]

    conf_matrix_v1 = confusion_matrix(labels, predictions_v1)
    conf_matrix_v2 = confusion_matrix(labels, predictions_v2)
    conf_matrix_v3 = confusion_matrix(labels, predictions_v3)

    print("Classification Report for MSO_V1:")
    print(classification_report(labels, predictions_v1, target_names=["Negative", "Positive"], zero_division=0))

    print("\nClassification Report for MSO_V2:")
    print(classification_report(labels, predictions_v2, target_names=["Negative", "Positive"], zero_division=0))

    print("\nClassification Report for MSO_V3:")
    print(classification_report(labels, predictions_v3, target_names=["Negative", "Positive"], zero_division=0))

    plot_confusion_matrices_side_by_side(
        conf_matrices=[conf_matrix_v1, conf_matrix_v2, conf_matrix_v3],
        model_names=["mso_v1", "mso_v2", "mso_v3"],
        class_names=["Negative", "Positive"]
    )


def plot_confusion_matrices_side_by_side(conf_matrices: List[np.ndarray], model_names: List[str], class_names: List[str]) -> None: 
    """
    Plots multiple confusion matrices side by side.

    Args:
        conf_matrices (list): List of confusion matrices (numpy arrays).
        model_names (list): List of model names corresponding to each confusion matrix.
        class_names (list): List of class names (e.g., ["Negative", "Positive"]).

    Returns:
        None: Displays the confusion matrices in a single figure.
    """
    num_matrices = len(conf_matrices)
    plt.figure(figsize=(6 * num_matrices, 6))

    for i, (conf_matrix, model_name) in enumerate(zip(conf_matrices, model_names)):
        plt.subplot(1, num_matrices, i + 1)
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        plt.title(f"Confusion Matrix for {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")

    plt.tight_layout()
    plt.show()


def calculate_iou(predicted_mask: np.ndarray, annotation_mask: np.ndarray) -> float: 
    """
    Calculates the Intersection over Union (IoU) between a predicted mask and an annotation mask.
    Handles cases where images have no annotations (negative images).

    Args:
        predicted_mask (np.ndarray): Binary mask predicted by the model (values 0 or 1).
        annotation_mask (np.ndarray): Binary mask annotated by the expert, or empty for negative images.

    Returns:
        float: IoU value, or NaN if masks are invalid.
    """
    if annotation_mask is None or predicted_mask is None:
        return float('nan')  

    if predicted_mask.shape != annotation_mask.shape:
        raise ValueError("Predicted mask and annotation mask must have the same shape.")

    predicted_mask = (predicted_mask > 0).astype(int)
    annotation_mask = (annotation_mask > 0).astype(int)

    intersection = np.logical_and(predicted_mask, annotation_mask).sum()
    union = np.logical_or(predicted_mask, annotation_mask).sum()

    if union == 0:
        return 1.0  

    iou = intersection / union
    return iou


def calculate_dice(predicted_mask: np.ndarray, annotation_mask: np.ndarray) -> float: 
    """
    Calculates the Dice Similarity Coefficient (DICE) between the predicted mask and the annotation mask.
    Handles cases where images have no annotations (negative images).

    Args:
        predicted_mask (np.ndarray): Binary mask predicted by the model (values 0 or 1).
        annotation_mask (np.ndarray): Binary mask annotated by the expert, or empty for negative images.

    Returns:
        float: DICE value, or NaN if masks are invalid.
    """
    if predicted_mask is None or annotation_mask is None:
        return float('nan')  

    if predicted_mask.shape != annotation_mask.shape:
        raise ValueError("Predicted mask and annotation mask must have the same shape.")

    predicted_mask = (predicted_mask > 0).astype(int)
    annotation_mask = (annotation_mask > 0).astype(int)

    predicted_sum = predicted_mask.sum()
    annotation_sum = annotation_mask.sum()

    if predicted_sum == 0 and annotation_sum == 0:
        return 1.0  

    if annotation_sum == 0 and predicted_sum > 0:
        return 0.0  

    intersection = np.logical_and(predicted_mask, annotation_mask).sum()

    dice = (2 * intersection) / (predicted_sum + annotation_sum)
    return dice


def display_statistics(results: List[Dict[str, Any]]) -> None: 
    """
    Displays statistics (mean, median, etc.) for IoU and DICE predictions for all three models.

    Args:
        results (list): List of dictionaries containing IoU and DICE values.

    Returns:
        None: Prints the statistics.
    """
    # Coletar os valores de IoU e DICE para os três modelos
    iou_v1_values = [result["iou_v1"] for result in results if not np.isnan(result["iou_v1"])]
    iou_v2_values = [result["iou_v2"] for result in results if not np.isnan(result["iou_v2"])]
    iou_v3_values = [result["iou_v3"] for result in results if not np.isnan(result["iou_v3"])]

    dice_v1_values = [result["dice_v1"] for result in results if not np.isnan(result["dice_v1"])]
    dice_v2_values = [result["dice_v2"] for result in results if not np.isnan(result["dice_v2"])]
    dice_v3_values = [result["dice_v3"] for result in results if not np.isnan(result["dice_v3"])]

    # Estatísticas para IOU
    print("IoU Statistics for MSO_V1:")
    print(f"Mean IoU: {np.mean(iou_v1_values):.4f}")
    print(f"Median IoU: {np.median(iou_v1_values):.4f}")
    print(f"Standard Deviation: {np.std(iou_v1_values):.4f}")

    print("\nIoU Statistics for MSO_V2:")
    print(f"Mean IoU: {np.mean(iou_v2_values):.4f}")
    print(f"Median IoU: {np.median(iou_v2_values):.4f}")
    print(f"Standard Deviation: {np.std(iou_v2_values):.4f}")

    print("\nIoU Statistics for MSO_V3:")
    print(f"Mean IoU: {np.mean(iou_v3_values):.4f}")
    print(f"Median IoU: {np.median(iou_v3_values):.4f}")
    print(f"Standard Deviation: {np.std(iou_v3_values):.4f}")

    # Estatísticas para DICE
    print("\nDICE Statistics for MSO_V1:")
    print(f"Mean DICE: {np.mean(dice_v1_values):.4f}")
    print(f"Median DICE: {np.median(dice_v1_values):.4f}")
    print(f"Standard Deviation: {np.std(dice_v1_values):.4f}")

    print("\nDICE Statistics for MSO_V2:")
    print(f"Mean DICE: {np.mean(dice_v2_values):.4f}")
    print(f"Median DICE: {np.median(dice_v2_values):.4f}")
    print(f"Standard Deviation: {np.std(dice_v2_values):.4f}")

    print("\nDICE Statistics for MSO_V3:")
    print(f"Mean DICE: {np.mean(dice_v3_values):.4f}")
    print(f"Median DICE: {np.median(dice_v3_values):.4f}")
    print(f"Standard Deviation: {np.std(dice_v3_values):.4f}")


def calculate_map(results: List[Dict[str, Any]], iou_thresholds: List[float] = [0.5]) -> Dict[str, Dict[str, float]]: 
    """ 
    Calculates the Mean Average Precision (mAP) for the predictions of each model at specified IoU thresholds. 
 
    Args: 
        results (List[Dict[str, Any]]): List of dictionaries containing predictions and annotations. 
        iou_thresholds (List[float]): List of IoU thresholds to consider for mAP calculation. 
 
    Returns: 
        Dict[str, Dict[str, float]]: Dictionary containing mAP values for each model at specified IoU thresholds. 
    """ 
    def compute_precision_recall(predictions, annotations, scores, iou_threshold): 
        precisions = [] 
        recalls = [] 
        total_positives = sum(1 for mask in annotations if mask is not None and mask.sum() > 0) 
 
        for threshold in np.linspace(0, 1, num=101):   
            true_positives = 0 
            false_positives = 0 
             
            for pred_mask, annotation_mask, score in zip(predictions, annotations, scores): 
                if annotation_mask is None or annotation_mask.sum() == 0: 
                    if pred_mask.sum() > 0 and score >= threshold: 
                        false_positives += 1 
                    continue 
                 
                iou = calculate_iou(pred_mask, annotation_mask) 
                if iou >= iou_threshold and score >= threshold: 
                    true_positives += 1 
                elif score >= threshold: 
                    false_positives += 1 
             
            precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0 
            recall = true_positives / total_positives if total_positives > 0 else 0 
 
            precisions.append(precision) 
            recalls.append(recall) 
 
        if not precisions or not recalls: 
            return [0], [0] 
 
        return precisions, recalls 
 
    # Coletar as predições e anotações para os três modelos
    predictions_v1 = [result.get("mask_v1", np.zeros_like(result.get("annotation_mask", np.array([])))) for result in results] 
    predictions_v2 = [result.get("mask_v2", np.zeros_like(result.get("annotation_mask", np.array([])))) for result in results] 
    predictions_v3 = [result.get("mask_v3", np.zeros_like(result.get("annotation_mask", np.array([])))) for result in results] 
    annotations = [result.get("annotation_mask", None) for result in results] 
 
    scores_v1 = [0.9] * len(results)   
    scores_v2 = [0.9] * len(results)   
    scores_v3 = [0.9] * len(results)   
 
    map_values = {"mAP_v1": {}, "mAP_v2": {}, "mAP_v3": {}} 
 
    for iou_threshold in iou_thresholds: 
        precisions_v1, recalls_v1 = compute_precision_recall(predictions_v1, annotations, scores_v1, iou_threshold) 
        precisions_v2, recalls_v2 = compute_precision_recall(predictions_v2, annotations, scores_v2, iou_threshold) 
        precisions_v3, recalls_v3 = compute_precision_recall(predictions_v3, annotations, scores_v3, iou_threshold) 
 
        recalls_v1, precisions_v1 = zip(*sorted(zip(recalls_v1, precisions_v1))) 
        recalls_v2, precisions_v2 = zip(*sorted(zip(recalls_v2, precisions_v2))) 
        recalls_v3, precisions_v3 = zip(*sorted(zip(recalls_v3, precisions_v3))) 
 
        map_v1 = auc(recalls_v1, precisions_v1) 
        map_v2 = auc(recalls_v2, precisions_v2) 
        map_v3 = auc(recalls_v3, precisions_v3) 
 
        map_values["mAP_v1"][f"mAP@{int(iou_threshold * 100)}"] = map_v1 
        map_values["mAP_v2"][f"mAP@{int(iou_threshold * 100)}"] = map_v2 
        map_values["mAP_v3"][f"mAP@{int(iou_threshold * 100)}"] = map_v3 
 
    return map_values 

## Visualização das predições

In [ ]:
results = generate_predictions(all_images, mso_v1, mso_v2, mso_v3)

positive_results = [result for result in results if result["annotation_mask"] is not None]
negative_results = [result for result in results if result["annotation_mask"] is None]

ordered_results = positive_results + negative_results

plot_image_comparisons(ordered_results)

## Avaliação das métricas

In [ ]:
evaluate_predictions(results, all_labels)
display_statistics(results)

map_results = calculate_map(results, iou_thresholds=[0.5, 0.9])
print("\nmAP Results:")
for threshold, value in map_results["mAP_v1"].items():
    print(f"MSO_V1 - {threshold}: {value:.4f}")
for threshold, value in map_results["mAP_v2"].items():
    print(f"MSO_V2 - {threshold}: {value:.4f}")
for threshold, value in map_results["mAP_v3"].items():
    print(f"MSO_V3 - {threshold}: {value:.4f}")

In [ ]:
!jupyter nbconvert --to html Pred_PUCLegado_MSOV1_MSOV2_MSOV3.ipynb

In [ ]:
import os
import zipfile

# Path to the folder you want to zip
folder_to_zip = 'features'  # Replace with your folder path
# Name of the output ZIP file
output_zip = 'my_folder_archive.zip'  # Replace with your desired ZIP file name

# Create a ZIP file
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for foldername, subfolders, filenames in os.walk(folder_to_zip):
        for filename in filenames:
            # Get the full file path
            file_path = os.path.join(foldername, filename)
            # Add the file to the ZIP archive with a relative path
            arcname = os.path.relpath(file_path, folder_to_zip)
            zipf.write(file_path, arcname)
            print(f"Adding '{file_path}' as '{arcname}' to the ZIP archive.")
    
    print(f"Folder '{folder_to_zip}' has been zipped as '{output_zip}'.")